In [ ]:
#
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from fractions import Fraction
import csv
import json
import mlbstatsapi
import os
import pytz
import re
import requests
import statsapi
import requests
from bs4 import BeautifulSoup
import json
import numpy as np
import matplotlib.pyplot as plt

import importlib
import script

# Reload the script after making changes
importlib.reload(script)

bvp_data = script.read_json_list('data/batter_vs_pitcher_data.json')

for a in bvp_data:
    hr = a.get('all_HR_record', '')
    hr_analysis_dict = script.analyze_score_sequence(hr)
    a.update({'all_HR_analysis': hr_analysis_dict})
    h = a.get('all_H_record', '')
    h_analysis_dict = script.analyze_score_sequence(h)
    a.update({'all_H_analysis': h_analysis_dict})
    rbi = a.get('all_RBI_record', '')
    rbi_analysis_dict = script.analyze_score_sequence(rbi)
    a.update({'all_RBI_analysis': rbi_analysis_dict})

script.save_to_json(bvp_data, "batter_vs_pitcher_data")

Archived existing file to data/archived_data/new_bvp_2025-08-24.json
Today's data saved to data/new_bvp.json


In [1]:

import json
import math
import os
from typing import Any, Dict, List, Union

import pandas as pd


def flatten_record(
    obj: Any,
    parent_key: str = "",
    out: Dict[str, Any] = None,
    sep: str = ".",
    list_max_items: int = 50,
    join_list_threshold: int = 15,
) -> Dict[str, Any]:
    """
    Recursively flattens a mixed JSON structure (dicts / lists / scalars) into a flat dict
    suitable for turning into a single DataFrame row.

    Rules:
      - Dict: key paths joined by 'sep'.
      - List of scalars (ints / floats / strings / None):
          * If len(list) <= join_list_threshold -> a single delimited string column.
          * Else -> numbered columns: key.0, key.1, ...
      - List of dicts:
          * Each element gets its own prefix: key[i].subkey
            (capped at list_max_items; remainder ignored to prevent column explosion)
      - None preserved (pandas will show NaN).
    """
    if out is None:
        out = {}

    def add(k, v):
        out[k] = v

    if isinstance(obj, dict):
        for k, v in obj.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            flatten_record(v, new_key, out, sep, list_max_items, join_list_threshold)
    elif isinstance(obj, list):
        # Classify list contents
        if not obj:
            add(parent_key, "")
            return out

        all_scalars = all(
            (x is None or isinstance(x, (str, int, float, bool))) for x in obj
        )
        all_dicts = all(isinstance(x, dict) for x in obj)

        if all_scalars:
            if len(obj) <= join_list_threshold:
                # Small scalar list: join into one cell
                joined = "|".join("" if v is None else str(v) for v in obj)
                add(parent_key, joined)
            else:
                # Large scalar list: separate columns
                for i, v in enumerate(obj[:list_max_items]):
                    add(f"{parent_key}{sep}{i}", v)
                if len(obj) > list_max_items:
                    add(f"{parent_key}{sep}__truncated_count", len(obj))
        elif all_dicts:
            for i, item in enumerate(obj[:list_max_items]):
                idx_key = f"{parent_key}[{i}]"
                flatten_record(
                    item,
                    idx_key,
                    out,
                    sep=sep,
                    list_max_items=list_max_items,
                    join_list_threshold=join_list_threshold,
                )
            if len(obj) > list_max_items:
                add(f"{parent_key}{sep}__truncated_count", len(obj))
        else:
            # Mixed list: store JSON string
            add(parent_key, json.dumps(obj, ensure_ascii=False))
    else:
        # Scalar
        if isinstance(obj, float):
            # Normalize NaN/inf for Excel
            if math.isnan(obj) or math.isinf(obj):
                obj = None
        add(parent_key, obj)
    return out


def json_list_to_flat_rows(data: Union[List, Dict]) -> List[Dict[str, Any]]:
    """
    Accepts either:
      - A list of records (expected)
      - A dict (will treat each value if values are lists / dicts)
    Returns list of flattened row dicts.
    """
    rows = []
    if isinstance(data, list):
        for rec in data:
            rows.append(flatten_record(rec))
    elif isinstance(data, dict):
        # Try to treat each value as a row (if they look like dicts)
        for k, v in data.items():
            base = {"__top_level_key": k}
            flat = flatten_record(v)
            base.update(flat)
            rows.append(base)
    else:
        raise ValueError("Top-level JSON must be list or dict.")
    return rows


def export_bvp_json_to_excel(
    input_path: str,
    output_path: str = None,
    sheet_name: str = "data",
    sep: str = ".",
    list_max_items: int = 50,
    join_list_threshold: int = 15,
) -> str:
    """
    Reads the JSON file at input_path (expected: list of dicts),
    flattens nested structures, and writes an Excel workbook.

    Parameters:
      input_path: path to JSON file
      output_path: output .xlsx path (defaults to same stem + '_flat.xlsx')
      sheet_name: worksheet name
      sep: separator used in flattened column names
      list_max_items: max elements from large lists / list of dicts to expand
      join_list_threshold: scalar list length threshold to join vs expand

    Returns:
      The output Excel path.
    """
    if output_path is None:
        base, _ = os.path.splitext(input_path)
        output_path = f"{base}_flat.xlsx"

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for rec in (data if isinstance(data, list) else [data]):
        flat = flatten_record(
            rec,
            sep=sep,
            list_max_items=list_max_items,
            join_list_threshold=join_list_threshold,
        )
        rows.append(flat)

    df = pd.DataFrame(rows)

    # Optional: stable column ordering (alphabetical)
    df = df.reindex(sorted(df.columns), axis=1)

    # Write Excel
    # Using xlsxwriter for speed; fallback to openpyxl if desired
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

    return output_path



src = "data/new_bvp.json"
out = export_bvp_json_to_excel(src)
print(f"Exported to {out}")

Exported to data/new_bvp_flat.xlsx
